# 03 — Tratamento dos Dados Meteorológicos

## Objetivo

Realizar o inventário, diagnóstico, tratamento e consolidação
dos dados meteorológicos disponibilizados pelo INMET.

## Recorte geográfico

Santos e os nove municípios da Baixada Santista.

## Recorte temporal

2021 a 2026.

O ano de 2026 será considerado parcial, pois ainda não está
encerrado.

## Etapas

1. Inventário dos arquivos ZIP.
2. Identificação das estações meteorológicas.
3. Seleção das estações pertencentes à área de estudo.
4. Diagnóstico de qualidade.
5. Padronização e consolidação.
6. Identificação de valores ausentes e inconsistências.
7. Exportação das bases processadas.

In [1]:
import pandas as pd
import numpy as np

import zipfile
import io
import re
import unicodedata

from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [9]:
from pathlib import Path

# O notebook está dentro de notebooks/
# Portanto, sua pasta pai é a raiz do projeto.

PASTA_PROJETO = Path.cwd().parent

# Caminhos dos dados
PASTA_RAW = PASTA_PROJETO / "data" / "raw" / "meteorologia"

PASTA_PROCESSED = (
    PASTA_PROJETO / "data" / "processed" / "meteorologia"
)

PASTA_QUALITY = (
    PASTA_PROJETO / "data" / "quality" / "meteorologia"
)

# Criar pastas de saída
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)
PASTA_QUALITY.mkdir(parents=True, exist_ok=True)

# Conferência
print("Raiz do projeto:", PASTA_PROJETO)
print("Pasta original:", PASTA_RAW)
print("Pasta processada:", PASTA_PROCESSED)
print("Pasta de qualidade:", PASTA_QUALITY)

print("\nPasta raw existe?", PASTA_RAW.exists())
print("Pasta meteorologia existe?", PASTA_RAW.is_dir())

Raiz do projeto: c:\Users\Lenovo\Desktop\porto-santos
Pasta original: c:\Users\Lenovo\Desktop\porto-santos\data\raw\meteorologia
Pasta processada: c:\Users\Lenovo\Desktop\porto-santos\data\processed\meteorologia
Pasta de qualidade: c:\Users\Lenovo\Desktop\porto-santos\data\quality\meteorologia

Pasta raw existe? True
Pasta meteorologia existe? True


In [8]:
from pathlib import Path

raiz = Path(r"C:\Users\Lenovo\Desktop\porto-santos")

print("RAIZ EXISTE:", raiz.exists())

print("\nCONTEÚDO DA PASTA DATA:")

for item in (raiz / "data").iterdir():
    print(repr(item.name), "| Diretório:", item.is_dir())

print("\nCONTEÚDO DA PASTA RAW:")

pasta_raw = raiz / "data" / "raw"

print("RAW EXISTE:", pasta_raw.exists())

if pasta_raw.exists():
    for item in pasta_raw.iterdir():
        print(repr(item.name), "| Diretório:", item.is_dir())

RAIZ EXISTE: True

CONTEÚDO DA PASTA DATA:
'interim' | Diretório: True
'processed' | Diretório: True
'quality' | Diretório: True
'raw' | Diretório: True

CONTEÚDO DA PASTA RAW:
RAW EXISTE: True
'atracacoes' | Diretório: True
'meterologia' | Diretório: True
'outros' | Diretório: True


In [10]:
arquivos_zip = sorted(PASTA_RAW.glob("*.zip"))

print(f"Quantidade de arquivos ZIP: {len(arquivos_zip)}")

for arquivo in arquivos_zip:
    print(arquivo.name)

Quantidade de arquivos ZIP: 6
2021.zip
2022.zip
2023.zip
2024.zip
2025.zip
2026.zip


In [11]:
def extrair_metadados(conteudo, nome_arquivo):

    metadados = pd.read_csv(
        io.BytesIO(conteudo),
        sep=";",
        header=None,
        nrows=8,
        encoding="latin1",
        dtype=str
    )

    dados = {}

    for _, linha in metadados.iterrows():

        chave = str(linha.iloc[0]).strip().replace(":", "")
        valor = str(linha.iloc[1]).strip()

        dados[chave] = valor

    def converter_coordenada(valor):

        valor = str(valor).strip().replace(",", ".")

        try:
            return float(valor)
        except ValueError:
            return np.nan

    resultado = {
        "arquivo": nome_arquivo,
        "regiao": dados.get("REGIAO"),
        "uf": dados.get("UF"),
        "estacao": dados.get("ESTACAO"),
        "codigo_estacao": dados.get("CODIGO (WMO)"),
        "latitude": converter_coordenada(
            dados.get("LATITUDE")
        ),
        "longitude": converter_coordenada(
            dados.get("LONGITUDE")
        ),
        "altitude": converter_coordenada(
            dados.get("ALTITUDE")
        ),
        "data_fundacao": dados.get("DATA DE FUNDACAO")
    }

    return resultado

In [12]:
inventario = []

for arquivo_zip in arquivos_zip:

    ano = arquivo_zip.stem

    print(f"Analisando {arquivo_zip.name}...")

    with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:

        arquivos_csv = [
            nome for nome in zip_ref.namelist()
            if nome.upper().endswith(".CSV")
        ]

        for nome_csv in arquivos_csv:

            try:
                conteudo = zip_ref.read(nome_csv)

                dados_estacao = extrair_metadados(
                    conteudo,
                    nome_csv
                )

                dados_estacao["ano_arquivo"] = ano

                inventario.append(dados_estacao)

            except Exception as erro:

                print(
                    f"Erro em {nome_csv}: {erro}"
                )

catalogo_estacoes = pd.DataFrame(inventario)

print("Inventário concluído!")
print("Total de arquivos identificados:", len(catalogo_estacoes))

display(catalogo_estacoes.head())

Analisando 2021.zip...
Analisando 2022.zip...
Analisando 2023.zip...
Analisando 2024.zip...
Analisando 2025.zip...
Analisando 2026.zip...
Inventário concluído!
Total de arquivos identificados: 4157


,arquivo,regiao,uf,estacao,codigo_estacao,latitude,longitude,altitude,data_fundacao,ano_arquivo
0,INMET_CO_DF_A001_BRASILIA_01-01-2021_A_31-12-2...,CO,DF,BRASILIA,A001,-15.789444,-47.925833,1160.96,07/05/00,2021
1,INMET_CO_DF_A042_BRAZLANDIA_01-01-2021_A_31-12...,CO,DF,BRAZLANDIA,A042,-15.599722,-48.131111,1143.00,19/07/17,2021
2,INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2021_A_...,CO,DF,AGUAS EMENDADAS,A045,-15.596491,-47.625801,1030.36,03/10/08,2021
3,INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2021_...,CO,DF,GAMA (PONTE ALTA),A046,-15.935278,-48.137500,990.00,01/10/14,2021
4,INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2021...,CO,DF,PARANOA (COOPA-DF),A047,-16.012222,-47.557417,1043.00,07/02/17,2021


In [21]:
import unicodedata

def normalizar_municipio(texto):
    texto = str(texto).strip().upper()

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    return texto


MUNICIPIOS_BAIXADA = [
    "SANTOS",
    "SAO VICENTE",
    "GUARUJA",
    "CUBATAO",
    "PRAIA GRANDE",
    "BERTIOGA",
    "MONGAGUA",
    "ITANHAEM",
    "PERUIBE"
]

catalogo_estacoes["municipio_norm"] = (
    catalogo_estacoes["municipio"]
    .apply(normalizar_municipio)
)

catalogo_estacoes["uf_norm"] = (
    catalogo_estacoes["uf"]
    .astype(str)
    .str.strip()
    .str.upper()
)

estacoes_baixada = catalogo_estacoes[
    (catalogo_estacoes["uf_norm"] == "SP") &
    (catalogo_estacoes["municipio_norm"].isin(MUNICIPIOS_BAIXADA))
].copy()

display(
    estacoes_baixada[
        ["codigo_estacao", "municipio", "uf", "ano_arquivo"]
    ].drop_duplicates().sort_values(
        ["municipio", "ano_arquivo"]
    )
)

KeyError: 'municipio'

In [ ]:
municipios_baixada = [
    "BERTIOGA",
    "CUBATAO",
    "GUARUJA",
    "ITANHAEM",
    "MONGAGUA",
    "PERUIBE",
    "PRAIA GRANDE",
    "SANTOS",
    "SAO VICENTE"
]

def normalizar_texto(texto):

    texto = str(texto).upper().strip()

    texto = unicodedata.normalize(
        "NFKD", texto
    ).encode("ASCII", "ignore").decode("ASCII")

    return texto


catalogo_estacoes["estacao_normalizada"] = (
    catalogo_estacoes["estacao"].apply(normalizar_texto)
)

catalogo_estacoes["municipio_candidato"] = (
    catalogo_estacoes["estacao_normalizada"].apply(
        lambda nome: next(
            (
                municipio
                for municipio in municipios_baixada
                if municipio in nome
            ),
            None
        )
    )
)

estacoes_candidatas = catalogo_estacoes[
    catalogo_estacoes["municipio_candidato"].notna()
].copy()

display(
    estacoes_candidatas[
        [
            "ano_arquivo",
            "estacao",
            "codigo_estacao",
            "municipio_candidato",
            "latitude",
            "longitude"
        ]
    ].sort_values(["ano_arquivo", "estacao"])
)

KeyError: 'municipio'

In [14]:
resumo_cobertura = (
    estacoes_candidatas
    .groupby(["ano_arquivo", "municipio_candidato"])
    .agg(
        quantidade_estacoes=("codigo_estacao", "nunique"),
        codigos=("codigo_estacao", lambda x: ", ".join(
            sorted(x.dropna().unique())
        ))
    )
    .reset_index()
)

display(resumo_cobertura)

,ano_arquivo,municipio_candidato,quantidade_estacoes,codigos
0,2021,BERTIOGA,1,A765
1,2021,SAO VICENTE,1,A889
2,2022,BERTIOGA,1,A765
3,2022,SAO VICENTE,1,A889
4,2023,BERTIOGA,1,A765
5,2023,SAO VICENTE,1,A889
6,2024,BERTIOGA,1,A765
7,2024,SAO VICENTE,1,A889
8,2025,BERTIOGA,1,A765
9,2025,SAO VICENTE,1,A889


In [15]:
resumo_cobertura = (
    estacoes_candidatas
    .groupby(["ano_arquivo", "municipio_candidato"])
    .agg(
        quantidade_estacoes=("codigo_estacao", "nunique"),
        codigos=("codigo_estacao", lambda x: ", ".join(
            sorted(x.dropna().unique())
        ))
    )
    .reset_index()
)

display(resumo_cobertura)

,ano_arquivo,municipio_candidato,quantidade_estacoes,codigos
0,2021,BERTIOGA,1,A765
1,2021,SAO VICENTE,1,A889
2,2022,BERTIOGA,1,A765
3,2022,SAO VICENTE,1,A889
4,2023,BERTIOGA,1,A765
5,2023,SAO VICENTE,1,A889
6,2024,BERTIOGA,1,A765
7,2024,SAO VICENTE,1,A889
8,2025,BERTIOGA,1,A765
9,2025,SAO VICENTE,1,A889


In [19]:
display(
    catalogo_estacoes[
        catalogo_estacoes["codigo_estacao"].isin(["A765", "A889"])
    ]
)

,arquivo,regiao,uf,estacao,codigo_estacao,latitude,longitude,altitude,data_fundacao,ano_arquivo,estacao_normalizada,municipio_candidato
487,INMET_SE_SP_A765_BERTIOGA_01-01-2021_A_31-12-2...,SE,SP,BERTIOGA,A765,-23.844722,-46.143333,5.0,01/02/17,2021,BERTIOGA,BERTIOGA
559,INMET_S_RS_A889_SAO VICENTE DO SUL_01-01-2021_...,S,RS,SAO VICENTE DO SUL,A889,-29.702222,-54.694444,134.0,06/04/16,2021,SAO VICENTE DO SUL,SAO VICENTE
1056,INMET_SE_SP_A765_BERTIOGA_01-01-2022_A_31-12-2...,SE,SP,BERTIOGA,A765,-23.844722,-46.143333,5.0,01/02/17,2022,BERTIOGA,BERTIOGA
1125,INMET_S_RS_A889_SAO VICENTE DO SUL_01-01-2022_...,S,RS,SAO VICENTE DO SUL,A889,-29.702222,-54.694444,134.0,06/04/16,2022,SAO VICENTE DO SUL,SAO VICENTE
1621,INMET_SE_SP_A765_BERTIOGA_01-01-2023_A_31-12-2...,SE,SP,BERTIOGA,A765,-23.844722,-46.143333,5.0,01/02/17,2023,BERTIOGA,BERTIOGA
1692,INMET_S_RS_A889_SAO VICENTE DO SUL_01-01-2023_...,S,RS,SAO VICENTE DO SUL,A889,-29.702222,-54.694444,134.0,06/04/16,2023,SAO VICENTE DO SUL,SAO VICENTE
2187,INMET_SE_SP_A765_BERTIOGA_01-01-2024_A_31-12-2...,SE,SP,BERTIOGA,A765,-23.844722,-46.143333,5.0,01/02/17,2024,BERTIOGA,BERTIOGA
2257,INMET_S_RS_A889_SAO VICENTE DO SUL_01-01-2024_...,S,RS,SAO VICENTE DO SUL,A889,-29.702222,-54.694444,134.0,06/04/16,2024,SAO VICENTE DO SUL,SAO VICENTE
2758,2025/INMET_SE_SP_A765_BERTIOGA_01-01-2025_A_31...,SE,SP,BERTIOGA,A765,-23.844722,-46.143333,5.0,01/02/17,2025,BERTIOGA,BERTIOGA
2828,2025/INMET_S_RS_A889_SAO VICENTE DO SUL_01-01-...,S,RS,SAO VICENTE DO SUL,A889,-29.702222,-54.694444,134.0,06/04/16,2025,SAO VICENTE DO SUL,SAO VICENTE
